In [ ]:
from huggingface_hub import login

# If you need authenticated access to the Hub (e.g. gated models or faster
# downloads), run this and paste your token when prompted.  You can create one
# at https://huggingface.co/settings/tokens.
#
# Alternatively you can set the environment variable manually:
#
import os
os.environ["HF_TOKEN"] = "<your_token_here>"
#
# Either approach populates `HF_TOKEN` so that subsequent `from_pretrained`
# calls use it.

login()

In [11]:
# !python -m pip uninstall -y transformers accelerate peft huggingface_hub tokenizers

# import os, site, glob, shutil

# for p in site.getsitepackages():
#     for pattern in [
#         "*transformers*",
#         "*accelerate*",
#         "*peft*",
#         "*huggingface_hub*",
#         "*tokenizers*",
#         "~*",
#     ]:
#         for path in glob.glob(os.path.join(p, pattern)):
#             try:
#                 if os.path.isdir(path):
#                     shutil.rmtree(path, ignore_errors=True)
#                 else:
#                     os.remove(path)
#                 print("removed:", path)
#             except Exception as e:
#                 print("skip:", path, e)

# !python -m pip install --upgrade pip setuptools wheel
# !python -m pip install --no-cache-dir "transformers==4.48.1" "accelerate>=0.26.0" "tokenizers<0.22,>=0.21" "huggingface_hub<1.0"

In [12]:
import json
import time
import gc
from pathlib import Path

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

In [13]:



def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda"), True
    return torch.device("cpu"), False


def create_prompt_and_target_length(tokenizer, num_input_tokens, num_output_tokens=64):
    base = "The quick brown fox jumps over the lazy dog. "
    enc = tokenizer.encode(base, add_special_tokens=False)
    repeat = (num_input_tokens // len(enc)) + 1
    long_text = (base * repeat).strip()
    inputs = tokenizer(
        long_text,
        return_tensors="pt",
        truncation=True,
        max_length=num_input_tokens
    )
    return inputs, num_output_tokens


def measure_ttft_and_throughput(model, tokenizer, inputs, device,
                                num_new_tokens=64, warmup=3):

    model.eval()

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    def gen_kwargs(max_tokens):
        kw = {
            "input_ids": input_ids,
            "max_new_tokens": max_tokens,
            "pad_token_id": tokenizer.eos_token_id or 0,
            "do_sample": False,
        }
        if attention_mask is not None:
            kw["attention_mask"] = attention_mask
        return kw

    # ----------------------
    # Warmup
    # ----------------------
    for _ in range(warmup):
        with torch.no_grad():
            _ = model.generate(**gen_kwargs(num_new_tokens))

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

    # ======================
    # 1️⃣ TTFT / Prefill phase
    # ======================
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        base_mem_prefill = torch.cuda.memory_allocated(device)

    start_ttft = time.perf_counter()
    with torch.no_grad():
        _ = model.generate(**gen_kwargs(1))
    if device.type == "cuda":
        torch.cuda.synchronize()

    ttft_sec = time.perf_counter() - start_ttft

    if device.type == "cuda":
        prefill_peak_mem = (torch.cuda.max_memory_allocated(device) - base_mem_prefill) / (1024**3)
    else:
        prefill_peak_mem = 0.0

    # ======================
    # 2️⃣ Full generation / Decoding phase
    # ======================
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        base_mem_decode = torch.cuda.memory_allocated(device)

    start_full = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**gen_kwargs(num_new_tokens))
    if device.type == "cuda":
        torch.cuda.synchronize()

    total_sec = time.perf_counter() - start_full

    num_generated = out.shape[1] - input_ids.shape[1]
    throughput = num_generated / total_sec if total_sec > 0 else 0.0

    if device.type == "cuda":
        decode_peak_mem = (torch.cuda.max_memory_allocated(device) - base_mem_decode) / (1024**3)
    else:
        decode_peak_mem = 0.0

    return {
        "ttft_sec": round(ttft_sec, 4),
        "tokens_generated": num_generated,
        "throughput_tokens_per_sec": round(throughput, 2),
        "peak_memory_prefill_gb": round(prefill_peak_mem, 4),
        "peak_memory_decoding_gb": round(decode_peak_mem, 4),
        # keep for backward-compat
        "peak_memory_gb": round(max(prefill_peak_mem, decode_peak_mem), 4),
    }


def run_benchmark(model_name, model_type, sequence_lengths,
                  num_output_tokens=64, warmup=3):

    device, is_cuda = get_device()
    print(f"\nDevice: {device}")
    print(f"Model: {model_name}")
    print(f"Type: {model_type}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # ----------------------
    # Load model — track peak memory
    # ----------------------
    load_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.float16 if is_cuda else torch.float32,
        "low_cpu_mem_usage": True,
    }
    if model_type == "jamba":
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        if hasattr(config, "use_mamba_kernels"):
            config.use_mamba_kernels = False
        if getattr(config, "mamba_config", None) is not None and hasattr(config.mamba_config, "use_mamba_kernels"):
            config.mamba_config.use_mamba_kernels = False
        load_kwargs["config"] = config

    if is_cuda:
        torch.cuda.reset_peak_memory_stats()
        base_mem_load = torch.cuda.memory_allocated(device)

    model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)

    if is_cuda:
        model = model.to(device)
        torch.cuda.synchronize()
        loading_peak_mem_gb = (torch.cuda.max_memory_allocated(device) - base_mem_load) / (1024**3)
    else:
        loading_peak_mem_gb = 0.0

    print(f"Model loading peak memory: {loading_peak_mem_gb:.2f} GB")

    results = {
        "model_name": model_name,
        "model_type": model_type,
        "device": str(device),
        "loading_peak_memory_gb": round(loading_peak_mem_gb, 4),
        "sequence_lengths": {},
    }

    # ----------------------
    # Benchmark loop
    # ----------------------
    for seq_len in sequence_lengths:
        print(f"\nBenchmarking sequence length: {seq_len}")

        inputs, num_out = create_prompt_and_target_length(
            tokenizer, seq_len, num_output_tokens
        )

        try:
            out = measure_ttft_and_throughput(
                model,
                tokenizer,
                inputs,
                device,
                num_new_tokens=num_out,
                warmup=warmup
            )

            results["sequence_lengths"][str(seq_len)] = out

            print(
                f"TTFT: {out['ttft_sec']:.3f}s | "
                f"Throughput: {out['throughput_tokens_per_sec']:.2f} tok/s | "
                f"Prefill mem: {out['peak_memory_prefill_gb']:.2f} GB | "
                f"Decode mem: {out['peak_memory_decoding_gb']:.2f} GB"
            )

        except Exception as e:
            results["sequence_lengths"][str(seq_len)] = {"error": str(e)}
            print("Error:", e)

    return model, tokenizer, results


In [14]:
RWKV_MODEL = "fla-hub/rwkv7-2.9B-world"
SEQUENCE_LENGTHS = [1024, 4096, 8192]
# BASELINE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
BASELINE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
NUM_OUTPUT_TOKENS = 64
WARMUP = 3
USE_8BIT = False  # Set to True if you hit OOM on a T4

In [15]:
!pip install 'accelerate>=0.26.0'

In [ ]:
model_rwkv, tokenizer_rwkv, results_rwkv = run_benchmark(
    RWKV_MODEL, "rwkv", SEQUENCE_LENGTHS,
    num_output_tokens=NUM_OUTPUT_TOKENS, warmup=WARMUP
)
print("RWKV done.")


Device: cpu
Model: fla-hub/rwkv7-2.9B-world
Type: rwkv


Encountered exception while importing fla: No module named 'fla'
You are using a model of type rwkv7 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
Encountered exception while importing fla: No module named 'fla'


ImportError: This modeling file requires the following packages that were not found in your environment: fla. Run `pip install fla`

In [ ]:
del model_rwkv
del tokenizer_rwkv
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memory cleared. Running Transformer baseline...")

In [ ]:
model_baseline, tokenizer_baseline, results_baseline = run_benchmark(
    BASELINE_MODEL, "transformer", SEQUENCE_LENGTHS,
    num_output_tokens=NUM_OUTPUT_TOKENS, warmup=WARMUP
)
print("Baseline done.")

In [ ]:
summary = {"rwkv": results_rwkv, "transformer_baseline": results_baseline}

Path("results").mkdir(exist_ok=True)
with open("results/summary_rwkv_vs_transformer.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved results/summary_rwkv_vs_transformer.json")

In [ ]:
def extract_metrics(record):
    if not record or "sequence_lengths" not in record:
        return [], [], [], [], []
    lengths, ttft, throughput, mem_prefill, mem_decode = [], [], [], [], []
    for L, v in sorted(record["sequence_lengths"].items(), key=lambda x: int(x[0])):
        if isinstance(v, dict) and "error" not in v:
            lengths.append(int(L))
            ttft.append(v.get("ttft_sec", 0))
            throughput.append(v.get("throughput_tokens_per_sec", 0))
            mem_prefill.append(v.get("peak_memory_prefill_gb", 0))
            mem_decode.append(v.get("peak_memory_decoding_gb", 0))
    return lengths, ttft, throughput, mem_prefill, mem_decode

r_len, r_ttft, r_tps, r_mem_pre, r_mem_dec = extract_metrics(results_rwkv)
b_len, b_ttft, b_tps, b_mem_pre, b_mem_dec = extract_metrics(results_baseline)

for label, res, lengths, ttft, tps, mem_pre, mem_dec in [
    ("RWKV", results_rwkv, r_len, r_ttft, r_tps, r_mem_pre, r_mem_dec),
    ("Transformer baseline", results_baseline, b_len, b_ttft, b_tps, b_mem_pre, b_mem_dec),
]:
    print(f"\n=== {label} ===")
    print(f"  Loading peak memory: {res.get('loading_peak_memory_gb', 0):.2f} GB")
    if lengths:
        print(f"  {'Length':>8} {'TTFT(s)':>10} {'Throughput':>12} {'Prefill(GB)':>12} {'Decode(GB)':>11}")
        for L, t, tp, mp, md in zip(lengths, ttft, tps, mem_pre, mem_dec):
            print(f"  {L:>8} {t:>10.3f} {tp:>12.2f} {mp:>12.4f} {md:>11.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Throughput ---
ax = axes[0]
if r_len:
    ax.plot(r_len, r_tps, "o-", label="RWKV-7-2.9B", color="C0")
if b_len:
    ax.plot(b_len, b_tps, "s-", label="Qwen2.5-3B", color="C1")
ax.set_xlabel("Input sequence length")
ax.set_ylabel("Throughput (tokens/sec)")
ax.set_title("Inference Throughput")
ax.legend()
ax.grid(True, alpha=0.3)

# --- TTFT ---
ax = axes[1]
if r_len:
    ax.plot(r_len, r_ttft, "o-", label="RWKV-7-2.9B", color="C0")
if b_len:
    ax.plot(b_len, b_ttft, "s-", label="Qwen2.5-3B", color="C1")
ax.set_xlabel("Input sequence length")
ax.set_ylabel("TTFT (sec)")
ax.set_title("Time to First Token")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Memory by phase (grouped bar chart) ---
ax = axes[2]
x = np.arange(len(r_len)) if r_len else np.arange(len(b_len))
x_labels = [str(l) for l in (r_len or b_len)]
width = 0.13

# RWKV: loading (constant), prefill, decode
r_load = [results_rwkv.get("loading_peak_memory_gb", 0)] * len(r_len)
b_load = [results_baseline.get("loading_peak_memory_gb", 0)] * len(b_len)

if r_len:
    ax.bar(x - 1.5*width, r_load,       width, label="RWKV load",    color="C0", alpha=0.4, hatch="//")
    ax.bar(x - 0.5*width, r_mem_pre,    width, label="RWKV prefill", color="C0", alpha=0.7)
    ax.bar(x + 0.5*width, r_mem_dec,    width, label="RWKV decode",  color="C0", alpha=1.0)
if b_len:
    ax.bar(x + 1.5*width, b_load,       width, label="Qwen load",    color="C1", alpha=0.4, hatch="//")
    ax.bar(x + 2.5*width, b_mem_pre,    width, label="Qwen prefill", color="C1", alpha=0.7)
    ax.bar(x + 3.5*width, b_mem_dec,    width, label="Qwen decode",  color="C1", alpha=1.0)

ax.set_xticks(x + width)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Input sequence length")
ax.set_ylabel("Peak memory (GB)")
ax.set_title("Peak Memory by Phase")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("results/rwkv_vs_transformer_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/rwkv_vs_transformer_plots.png")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create the figure and primary axis
fig, ax1 = plt.subplots(figsize=(12, 6))

x = np.arange(len(r_len)) if r_len else np.arange(len(b_len))
x_labels = [str(l) for l in (r_len or b_len)]
width = 0.13

# Loading memory (constant across sequence lengths)
r_load = [results_rwkv.get("loading_peak_memory_gb", 0)] * len(r_len) if r_len else []
b_load = [results_baseline.get("loading_peak_memory_gb", 0)] * len(b_len) if b_len else []

# --- 1. Memory by phase (grouped bar chart) on Primary Y-axis ---
if r_len:
    # Offset by -2.5, -1.5, -0.5 to center the 3 RWKV bars on the left
    ax1.bar(x - 2.5*width, r_load,    width, label="RWKV load",    color="C0", alpha=0.4, hatch="//")
    ax1.bar(x - 1.5*width, r_mem_pre, width, label="RWKV prefill", color="C0", alpha=0.7)
    ax1.bar(x - 0.5*width, r_mem_dec, width, label="RWKV decode",  color="C0", alpha=1.0)
if b_len:
    # Offset by +0.5, +1.5, +2.5 to center the 3 Transformer bars on the right
    ax1.bar(x + 0.5*width, b_load,    width, label="Qwen load",    color="C1", alpha=0.4, hatch="//")
    ax1.bar(x + 1.5*width, b_mem_pre, width, label="Qwen prefill", color="C1", alpha=0.7)
    ax1.bar(x + 2.5*width, b_mem_dec, width, label="Qwen decode",  color="C1", alpha=1.0)

ax1.set_xticks(x)
ax1.set_xticklabels(x_labels)
ax1.set_xlabel("Input sequence length", fontweight="bold")
ax1.set_ylabel("Peak memory (GB)", fontweight="bold")
ax1.grid(True, alpha=0.3, axis="y")

# --- 2. TTFT as Line Charts on Secondary Y-axis ---
ax2 = ax1.twinx()
if r_len:
    ax2.plot(x, r_ttft, "o-", label="RWKV-7 TTFT", color="#0343df", linewidth=2.5)
if b_len:
    ax2.plot(x, b_ttft, "s-", label="Qwen TTFT", color="#e50000", linewidth=2.5)

ax2.set_ylabel("TTFT (sec)", fontweight="bold")

# --- 3. Combine legends from both axes ---
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
# Place legend at upper left with 2 columns to keep it compact

# Find the highest memory value and add 35% more space to the top of the y-axis
max_mem = max(max(r_mem_dec) if r_len else 0, max(b_mem_dec) if b_len else 0)
ax1.set_ylim(0, max_mem * 2.5)

# Place legend at upper left with 2 columns to keep it compact
ax1.legend(
    handles1 + handles2, labels1 + labels2,
    loc="upper left",
    fontsize=10,
    ncol=2,
    framealpha=0.9
)

# ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left", fontsize=10, ncol=2, framealpha=0.9)

plt.title("Peak Memory by Phase & TTFT vs Sequence Length", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()

# Save and display
plt.savefig("results/rwkv_vs_transformer_memory_ttft.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/rwkv_vs_transformer_memory_ttft.png")